# 01 — Fitting basics

**Phenomenological Growth Models Lab**

By the end of this notebook you will have fitted a growth model to a real outbreak
curve, read off what its parameters mean, and developed a feel for how each one
changes the shape of the epidemic.

A *phenomenological* model describes the shape a curve takes without claiming to
represent transmission mechanistically. There is no susceptible compartment here, no
contact rate, no reproduction number falling out of the equations. That is a real
limitation — and also why these models are useful early in an outbreak, when you have
a case curve and very little else.

## Setup

These notebooks run against the environment pinned in this repository's `Project.toml`
and `Manifest.toml`. The cell below activates it and installs anything missing.

The first run downloads and precompiles the dependencies — a few minutes, once.
Every run after that starts in seconds.

Because the versions are pinned in `Manifest.toml`, you get the same package versions
these notebooks were written against, whatever else is installed on your machine. If you
skip this cell and rely on your global environment, newer versions may resolve
differently and the numbers below may not match.

In [ ]:
using Pkg

# Works whether the notebook was launched from the repository root or from notebooks/.
root = isfile("Project.toml") ? "." : ".."
Pkg.activate(root)
Pkg.instantiate()

println("Active project: ", Base.active_project())

In [ ]:
using GrowthFit
using Plots, Statistics, Printf, DelimitedFiles, Random

default(size = (760, 420), lw = 2, legend = :topright, framestyle = :box,
        guidefontsize = 10, titlefontsize = 11)

# Threads. Notebooks 03 and 05 bootstrap in parallel. A Jupyter kernel does
# NOT inherit `-t` from whatever launched Jupyter — the thread count is fixed
# by the kernel itself — so if this says 1, see the README for the one-off
# `IJulia.installkernel` step. Results are identical either way; only speed
# changes.
println("GrowthFit loaded. Threads available: ", Threads.nthreads())
Threads.nthreads() == 1 && println("  (single-threaded — notebooks 03 and 05 will be slower; see README)")


> **Tip.** The bootstrap in notebooks 03 and 05 runs in parallel. If you want it faster,
> restart Julia with `-t 8`. Results are identical either way — each replicate is seeded
> independently of the thread count.

### The data

Weekly confirmed measles cases from Jalisco, Mexico. Two columns: week index and case
count. The cell below uses the local copy if you cloned the repository, and downloads it
otherwise — so it works either way.

In [ ]:
function load_series(name)
    path = isfile(joinpath("data", name)) ? joinpath("data", name) :
           joinpath("..", "data", name)
    isfile(path) || error("Could not find $name. Run this notebook from inside the " *
                          "cloned repository so the data/ folder is reachable.")
    raw = readdlm(path, '\t', Float64)
    return raw[:, 1], raw[:, 2]          # time, incidence
end

timevect_all, ydata_all = load_series("jalisco_measles.txt")

@printf("%d weeks, peak of %.0f cases, %.0f cases total\n",
        length(ydata_all), maximum(ydata_all), sum(ydata_all))

Always plot the data before fitting anything.

In [ ]:
scatter(timevect_all, ydata_all,
        label = "observed", mc = :steelblue, ms = 4, msw = 0,
        xlabel = "week", ylabel = "weekly cases",
        title = "Measles, Jalisco (Mexico)")

A single wave: slow start, sharp rise, a peak, then decline. That shape — an
S-curve in *cumulative* cases, a single hump in *weekly* cases — is what every model
in this toolbox is built to capture.

### Calibration and evaluation periods

We will hold back the last few weeks. The model never sees them, which lets us check
later whether its forecasts were any good. Fitting to all the data and then declaring
a good fit tells you almost nothing.

In [ ]:
calibration_period = 40
forecast_horizon   = 3

timevect = timevect_all[1:calibration_period]
ydata    = ydata_all[1:calibration_period]

println("Calibrating on weeks $(Int(timevect[1]))–$(Int(timevect[end]))")
println("Holding back $forecast_horizon weeks for evaluation")

## The models

All seven models describe how *cumulative* cases $x$ grow over time. The package
reports the result as weekly incidence — the week-to-week differences.

| flag | Model | ODE |
| --- | --- | --- |
| `:exp` | Exponential Growth | $dx/dt = r x$ |
| `:ggm` | Generalized Growth | $dx/dt = r x^p$ |
| `:lm` | Logistic | $dx/dt = r x (1 - x/K)$ |
| `:glm` | Generalized Logistic | $dx/dt = r x^p (1 - x/K)$ |
| `:rich` | Richards | $dx/dt = r x (1 - (x/K)^a)$ |
| `:grm` | Generalized Richards | $dx/dt = r x^p (1 - (x/K)^a)$ |
| `:gom` | Gompertz | $dx/dt = r x e^{-a t}$ |

Four parameters appear:

- $r$ — **growth rate**. How fast cases accumulate early on.
- $p$ — **deceleration of growth**, between 0 and 1. At $p = 1$ growth is exponential;
  below 1 it is sub-exponential. Values near 0.5 correspond to roughly quadratic growth,
  which is what you see when transmission is spatially clustered rather than well-mixed.
- $K$ — **final epidemic size**. The total number of cases the curve saturates at.
- $a$ — a **shape parameter** controlling how sharply the curve turns over at the peak.

Note the nesting. Set $p = 1$ and $a = 1$ in the Generalized Richards Model and you get
the Logistic. Drop $K$ to infinity and you get the Generalized Growth Model. Each model
is a special case of a more flexible one, which is what makes the comparison in
notebook 02 meaningful.

You can inspect the bookkeeping directly:

In [ ]:
for flag in (:exp, :ggm, :lm, :glm, :rich, :grm, :gom)
    @printf("%-6s %-32s %d free parameter(s)\n",
            flag, MODEL_NAMES[flag], MODEL_NPARAMS[flag])
end

## Simulating before fitting

`solve_incidence` runs the ODE forward for parameters you choose. Before letting an
optimizer pick them, it is worth turning the knobs yourself — it makes the fitted
numbers mean something later.

The signature is `solve_incidence(flag, r, p, a, K, I0, timevect)`. Parameters a model
does not use are ignored, so passing `1.0` for them is fine.

In [ ]:
I0 = ydata[1]        # seed the ODE at the first observed count

curve = solve_incidence(:lm, 0.35, 1.0, 1.0, 4000.0, I0, timevect)

plot(timevect, curve, label = "logistic, r=0.35, K=4000",
     xlabel = "week", ylabel = "weekly cases")
scatter!(timevect, ydata, label = "observed", mc = :steelblue, ms = 4, msw = 0)

Not a good fit — but that is the point. Now vary one parameter at a time.

**$r$ controls how fast the epidemic takes off:**

In [ ]:
plt = scatter(timevect, ydata, label = "observed", mc = :grey70, ms = 3, msw = 0,
              xlabel = "week", ylabel = "weekly cases", title = "Varying r (K fixed at 7000)")
for r in (0.2, 0.35, 0.5, 0.8)
    c = solve_incidence(:lm, r, 1.0, 1.0, 7000.0, I0, timevect)
    c === nothing || plot!(plt, timevect, c, label = "r = $r")
end
plt

Higher $r$ pushes the peak earlier and makes it taller. The area under the curve
barely changes, because that is governed by $K$, not $r$.

**$K$ controls the total size:**

In [ ]:
plt = scatter(timevect, ydata, label = "observed", mc = :grey70, ms = 3, msw = 0,
              xlabel = "week", ylabel = "weekly cases", title = "Varying K (r fixed at 0.35)")
for K in (3000.0, 5000.0, 7000.0, 10000.0)
    c = solve_incidence(:lm, 0.35, 1.0, 1.0, K, I0, timevect)
    c === nothing || plot!(plt, timevect, c, label = "K = $(Int(K))")
end
plt

**$p$ controls whether growth is exponential or slower.** This one is easiest to see
on a log scale, where exponential growth is a straight line:

In [ ]:
plt = plot(xlabel = "week", ylabel = "weekly cases (log scale)", yscale = :log10,
           title = "Varying p in the Generalized Growth Model", legend = :bottomright)
for p in (0.6, 0.8, 0.9, 1.0)
    c = solve_incidence(:ggm, 0.35, p, 1.0, 1.0, I0, timevect[1:25])
    c === nothing || plot!(plt, timevect[1:25], max.(c, 0.1), label = "p = $p")
end
plt

At $p = 1$ the line is straight — pure exponential growth. Below 1 it bends downward:
growth is still accelerating, but more slowly than exponential. Most real outbreaks sit
somewhere in between, which is exactly why the Generalized Growth Model exists.

**Exercise.** Go back to the $r$ cell and try `r = 2.0`. What happens, and why does the
plot look wrong? (Hint: look at the $y$-axis, then think about what the logistic ODE does
when growth outruns $K$.)

## Fitting

Now let the optimizer choose. `fit_growth_model` maximizes the likelihood of the data
under an assumed error structure.

The `dist` argument matters more than beginners expect:

- `:normal` — least squares. Assumes errors have the same size everywhere, which is
  wrong for count data: a week with 500 cases is noisier in absolute terms than a week
  with 5.
- `:poisson` — variance equals the mean. Right shape, but real surveillance data is
  almost always *more* variable than Poisson allows.
- `:nb1` — negative binomial with $\mathrm{Var} = \mu + \alpha\mu$. The extra parameter
  $\alpha$ absorbs overdispersion. This is usually the honest choice for case counts.

We will use `:nb1` throughout.

In [ ]:
Random.seed!(1)     # pin the optimizer's random restarts

fit = fit_growth_model(:grm, timevect, ydata; dist = :nb1, n_restarts = 10)

The returned `FitResult` holds everything about the fit:

In [ ]:
@printf("model      : %s\n", MODEL_NAMES[fit.flag])
@printf("converged  : %s\n", fit.converged)
@printf("neg. loglik: %.2f\n", fit.objective)
println()
@printf("r (growth rate)      = %.4f\n", fit.r)
@printf("p (deceleration)     = %.4f\n", fit.p_exp)
@printf("a (shape)            = %.4f\n", fit.a)
@printf("K (final size)       = %.1f\n", fit.K)
@printf("alpha (overdispersion) = %.2f\n", fit.alpha)

In [ ]:
plot(timevect, fit.fitcurve, label = "GRM fit", lc = :crimson,
     xlabel = "week", ylabel = "weekly cases",
     title = "Generalized Richards Model, NB1 error")
scatter!(timevect, ydata, label = "observed", mc = :steelblue, ms = 4, msw = 0)

### Reading the numbers

Look at $p$. It came out well below 1, which says growth in Jalisco was clearly
sub-exponential — consistent with measles spreading through clustered under-vaccinated
communities rather than a well-mixed population.

$K$ is an estimate of the **final epidemic size**: the total cases this wave will
produce if nothing changes. Compare it to what had accumulated by the end of the
calibration window:

In [ ]:
@printf("Cases observed through week %d : %.0f\n", Int(timevect[end]), sum(ydata))
@printf("Estimated final size K         : %.0f\n", fit.K)
@printf("Implied still to come          : %.0f\n", fit.K - sum(ydata))

That single number is often the thing decision-makers actually want. Treat it with
care — notebook 03 puts a confidence interval on it, and notebook 04 explains why that
interval is wider than you might hope.

$\alpha$ is large, which confirms the counts are heavily overdispersed relative to
Poisson. Had we fitted with `dist = :poisson`, the prediction intervals would have come
out far too narrow.

### Try a different error structure

Same model, same data, three assumptions about noise:

In [ ]:
Random.seed!(1)
fits = Dict(d => fit_growth_model(:grm, timevect, ydata; dist = d, n_restarts = 10)
            for d in (:normal, :poisson, :nb1))

println("dist       r        p        a        K")
for d in (:normal, :poisson, :nb1)
    f = fits[d]
    @printf("%-9s %-8.4f %-8.4f %-8.4f %.0f\n", d, f.r, f.p_exp, f.a, f.K)
end

In [ ]:
plt = scatter(timevect, ydata, label = "observed", mc = :grey60, ms = 3, msw = 0,
              xlabel = "week", ylabel = "weekly cases", title = "Same model, three error structures")
for d in (:normal, :poisson, :nb1)
    plot!(plt, timevect, fits[d].fitcurve, label = String(d))
end
plt

Those are not small differences. Least squares lands somewhere quite distinct from
Poisson and NB1, which agree closely with each other — and the `:normal` curve visibly
misses the early weeks while tracking the decline.

This is a real property of the estimator, not a numerical accident. The error structure
decides **which observations get the most weight**:

- **Least squares** treats an error of 50 cases the same whether the week had 20 cases or
  700. The squared residuals near the peak are enormous compared with everything else, so
  the fit is effectively determined by the peak region.
- **Poisson and NB1** work on the log scale, so what matters is *relative* error. Missing a
  week of 20 cases by 10 counts for as much as missing a week of 500 by 250. The early
  low-count weeks therefore pull real weight.

Watch what happens to $p$ in particular. Under least squares it runs to its upper bound of
1 — the peak carries no information about early sub-exponential growth, so nothing holds it
down. Under NB1 the early weeks pull it to about 0.84. **$p$ is the parameter most sensitive
to the choice of error structure**, which is awkward, because $p$ is usually the parameter
people most want to interpret.

For epidemic case counts, `:nb1` is the defensible default: counts are overdispersed, and
you rarely want a fit that ignores the entire growth phase.

> **A note on versions.** Through GrowthFit v0.1.x, `:normal` fits were unreliable for a
> separate reason — the optimizer would stall and return its starting point. If you see a
> wildly different least-squares fit from the one here, check that you are on v0.2.0 or
> later. The package README documents what went wrong.